# Regional run QA: map + time series small multiples
Small multiples across the *full* regional-run datatrees (CESM2-WACCM, MIROC-ES2H, UKESM),
covering every scenario / variable / ensemble member found in each store.

Nested `#` GCM / `##` scenario headers, 2 grid figures per scenario
(rows=variable, cols=member) instead of one figure per var/member combo:

1. **Time series** -- single point, full record.
2. **Map** -- first day vs last day, side by side per member.



In [ ]:
import logging
import os

import dask
import frisky
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr
from dask.distributed import Client
from dask_array.xarray import register
from IPython.display import Markdown, display

register()
zarr.config.set({"async.concurrency": 128})

os.environ["FRISKY_SUMMARY"] = "off"
os.environ["FRISKY_DEATH_DUMP_DIR"] = ""

logging.getLogger("distributed.worker.memory").setLevel(logging.ERROR)

In [ ]:
client = frisky.hijack(Client(n_workers=12))
client

## Load the datatrees

In [ ]:
%%time
BUCKET = "carbonplan-scratch"
BRANCH = "full-regional-run"

PREFIXES = {
    "CESM2-WACCM": "srm/output/qa/CESM2-WACCM-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
    "MIROC-ES2H": "srm/output/qa/MIROC-ES2H-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
    "UKESM": "srm/output/qa/UKESM-ERA5-lat-31.0to-26.0_lon23.0to30.0.icechunk",
}


def open_datatree(bucket: str, prefix: str, branch: str = BRANCH) -> xr.DataTree:
    storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, from_env=True)
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session(branch=branch)
    return xr.open_datatree(session.store, engine="zarr", chunks="auto")


datatrees = {name: open_datatree(BUCKET, prefix) for name, prefix in PREFIXES.items()}

## Config

In [ ]:
POINT_LAT = -29.0
POINT_LON = 27.0

## Walk the tree



In [ ]:
def group_leaves(
    datatrees: dict[str, xr.DataTree],
) -> dict[tuple[str, str], dict[str, dict[str, xr.DataArray]]]:
    """Walk every model's tree and regroup as grouped[(model, scenario)][variable][member] = da."""
    grouped: dict[tuple[str, str], dict[str, dict[str, xr.DataArray]]] = {}
    for model_name, dt in datatrees.items():
        for node in dt.subtree:
            if not node.is_leaf:
                continue
            ds_node = node.to_dataset()
            if not ds_node.data_vars:
                continue
            parts = node.path.strip("/").split("/")
            # debiased_coarse mirrors the scenario groups one level down
            # (debiased_coarse/<scenario>/<variable>/<member>)
            # dropping all but the last one.
            if parts[0] == "debiased_coarse":
                scenario = f"debiased_coarse/{parts[1]}" if len(parts) > 1 else "debiased_coarse"
                variable, member = (parts[2:4] + [None, None])[:2]
            else:
                scenario, variable, member = (parts + [None, None, None])[:3]
            for da in ds_node.data_vars.values():
                grouped.setdefault((model_name, scenario), {}).setdefault(variable, {})[member] = da
    return grouped


grouped = group_leaves(datatrees)
list(grouped.keys())

## Plot functions

In [ ]:
def _mark_empty(ax: plt.Axes) -> None:
    ax.set_xticks([])
    ax.set_yticks([])


def _label_grid_cell(
    ax: plt.Axes, i: int, j: int, row_label: str, col_label: str, title_fontsize: int = 8
) -> None:
    if j == 0:
        ax.set_ylabel(row_label, fontsize=8)
    if i == 0:
        ax.set_title(col_label, fontsize=title_fontsize)


def plot_timeseries_grid(
    ts: dict[tuple[str, str], xr.DataArray],
    variables: list[str],
    members: list[str],
    title: str,
) -> plt.Figure:
    n_rows, n_cols = len(variables), len(members)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(2.8 * n_cols, 1.8 * n_rows), sharex="row", squeeze=False
    )

    for i, var in enumerate(variables):
        for j, member in enumerate(members):
            ax = axes[i, j]
            _label_grid_cell(ax, i, j, var, member)

            da = ts.get((var, member))
            if da is None:
                _mark_empty(ax)
                continue
            ax.plot(da["time"].values, da.values, lw=0.5)
            ax.tick_params(labelsize=6)

    fig.suptitle(f"{title} -- time series @ lat={POINT_LAT}, lon={POINT_LON}")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


def plot_map_grid(
    map_first: dict[tuple[str, str], xr.DataArray],
    map_last: dict[tuple[str, str], xr.DataArray],
    variables: list[str],
    members: list[str],
    title: str,
    robust: bool = True,
) -> plt.Figure:
    days = [("first", map_first), ("last", map_last)]
    n_rows, n_cols = len(variables), len(members) * len(days)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(1.4 * n_cols, 2.0 * n_rows), squeeze=False, layout="constrained"
    )

    for i, var in enumerate(variables):
        row_das = [
            da
            for day_label, maps in days
            for member in members
            if (da := maps.get((var, member))) is not None
        ]
        if row_das:
            pooled = np.concatenate([d.values.ravel() for d in row_das])
            if robust:
                vmin, vmax = np.nanpercentile(pooled, [2, 98])
            else:
                vmin, vmax = np.nanmin(pooled), np.nanmax(pooled)
            units = row_das[0].attrs.get("units")
        else:
            vmin = vmax = units = None

        row_im = None
        for j, member in enumerate(members):
            for d, (day_label, maps) in enumerate(days):
                col = j * len(days) + d
                ax = axes[i, col]
                ax.set_xticks([])
                ax.set_yticks([])
                _label_grid_cell(ax, i, col, var, f"{member}\n{day_label}", title_fontsize=7)

                da = maps.get((var, member))
                if da is None:
                    _mark_empty(ax)
                    continue
                lat, lon = da["lat"].values, da["lon"].values
                extent = [lon.min(), lon.max(), lat.min(), lat.max()]
                origin = "lower" if lat[0] < lat[-1] else "upper"
                row_im = ax.imshow(
                    da.values, vmin=vmin, vmax=vmax, extent=extent, origin=origin, aspect="auto"
                )

                ax.set_xlabel(str(da["time"].values)[:10], fontsize=6)

        if row_im is not None:
            cbar = fig.colorbar(row_im, ax=axes[i, :].tolist(), fraction=0.02, pad=0.01)
            if units:
                cbar.set_label(units, fontsize=7)

    fig.suptitle(f"{title} -- map (first vs last day)")
    return fig

## Batched select + compute

Build every lazy selection first, `dask.compute` them in one shot, then plot from
the already-loaded results -- one compute round-trip total, not one per panel.

In [ ]:
selections = {}
for (model_name, scenario), by_var in grouped.items():
    for variable, by_member in by_var.items():
        for member, da in by_member.items():
            key = (model_name, scenario, variable, member)
            selections[(*key, "ts")] = da.sel(lat=POINT_LAT, lon=POINT_LON, method="nearest")
            selections[(*key, "map_first")] = da.isel(time=0)
            selections[(*key, "map_last")] = da.isel(time=-1)

loaded = dict(zip(selections.keys(), dask.compute(*selections.values())))
len(selections)

## Plot per (model, scenario)

`#` header per GCM, `##` header per scenario underneath it

In [ ]:
last_model = None
for model_name, scenario in sorted(grouped.keys()):
    if model_name != last_model:
        display(Markdown(f"# {model_name}"))
        last_model = model_name

    title = f"{model_name} / {scenario}"
    by_var = grouped[(model_name, scenario)]
    variables = sorted(by_var.keys())
    members = sorted({m for by_member in by_var.values() for m in by_member})

    ts, map_first, map_last = {}, {}, {}
    for variable in variables:
        for member in by_var[variable]:
            key = (variable, member)
            ts[key] = loaded[(model_name, scenario, variable, member, "ts")]
            map_first[key] = loaded[(model_name, scenario, variable, member, "map_first")]
            map_last[key] = loaded[(model_name, scenario, variable, member, "map_last")]

    display(Markdown(f"## {scenario}"))

    fig_ts = plot_timeseries_grid(ts, variables, members, title)
    display(fig_ts)
    plt.close(fig_ts)

    fig_map = plot_map_grid(map_first, map_last, variables, members, title)
    display(fig_map)
    plt.close(fig_map)